# 4.3. Augmented Data for Image Classification
## This is Part 4.3. in the official paper, image synthesis method: CycleGAN

We use official Pytorch implementation of CycleGAN: https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

This notebook generates synthetic images of sick leaves from healthy leaves and uses this data for classification.

In [ ]:
!pip install torch torchvision matplotlib numpy scikit-learn

  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
import os
import shutil
import sys
import pandas as pd
import torch
import torch.optim as optim
from torchvision import  datasets, transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn as nn
import glob


from torchvision.models import vgg19, VGG19_Weights
import torch.nn.functional as F
from torch.utils.data import DataLoader
import time
import numpy as np

import random
import copy
import cv2

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_COLAB

True

In [ ]:
# Set project path based on the environment
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/My Drive"):
      drive.mount('/content/drive')
    else:
      print("Drive already mounted")

    project_path = "/content/drive/My Drive/4.3. CycleGAN/"

else:
    project_path = "../"

os.chdir(project_path)

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
device

device(type='cuda')

Cloning the official repository

In [ ]:
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2516, done.
remote: Total 2516 (delta 0), reused 0 (delta 0), pack-reused 2516 (from 1)
Receiving objects: 100% (2516/2516), 8.20 MiB | 17.24 MiB/s, done.
Resolving deltas: 100% (1575/1575), done.
/content/drive/MyDrive/4.3. CycleGAN/pytorch-CycleGAN-and-pix2pix
  Using cached dominate-2.9.1-py2.py3-none-any.whl.metadata (13 kB)
  Using cached visdom-0.2.4.tar.gz (1.4 MB)
  Preparing metadata (setup.py) ... done
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidi

In [ ]:
%cd pytorch-CycleGAN-and-pix2pix

/content/drive/.shortcut-targets-by-id/1dVqJd7Cl8ksqLtFrrVzyjt6DPdI4-0c_/4.3. CycleGAN/pytorch-CycleGAN-and-pix2pix


In [ ]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.3 MB/s eta 0:00:00
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408196 s

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"aminaism","key":"fb952e06356e9e6c35565dc1d9d113e5"}'}

Downloading dataset and setting up the training/testing sets

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
%cd datasets/
!kaggle competitions download -c plant-pathology-2020-fgvc7
!unzip plant-pathology-2020-fgvc7.zip -d plant_pathology_data

/content/drive/MyDrive/4.3. CycleGAN/pytorch-CycleGAN-and-pix2pix/datasets
Archive:  plant-pathology-2020-fgvc7.zip
  inflating: plant_pathology_data/images/Test_0.jpg  
  inflating: plant_pathology_data/images/Test_1.jpg  
  inflating: plant_pathology_data/images/Test_10.jpg  
  inflating: plant_pathology_data/images/Test_100.jpg  
  inflating: plant_pathology_data/images/Test_1000.jpg  
  inflating: plant_pathology_data/images/Test_1001.jpg  
  inflating: plant_pathology_data/images/Test_1002.jpg  
  inflating: plant_pathology_data/images/Test_1003.jpg  
  inflating: plant_pathology_data/images/Test_1004.jpg  
  inflating: plant_pathology_data/images/Test_1005.jpg  
  inflating: plant_pathology_data/images/Test_1006.jpg  
  inflating: plant_pathology_data/images/Test_1007.jpg  
  inflating: plant_pathology_data/images/Test_1008.jpg  
  inflating: plant_pathology_data/images/Test_1009.jpg  
  inflating: plant_pathology_data/images/Test_101.jpg  
  inflating: plant_pathology_data/image

In [ ]:
# Load the CSV file
csv_path = "plant_pathology_data/train.csv"
df = pd.read_csv(csv_path)

# Define source and destination directories
source_dir = "plant_pathology_data/images"
trainA_dir = "plant_pathology_data/trainA"
trainB_dir = "plant_pathology_data/trainB"
testA_dir = "plant_pathology_data/testA"
testB_dir = "plant_pathology_data/testB"

# Create directories if they don't exist
for dir_path in [trainA_dir, trainB_dir, testA_dir, testB_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Shuffle dataset for random selection
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Initialize counters
train_healthy_count = 0
train_sick_selected = False
test_healthy_count = 0
test_sick_count = 0

# Track used image IDs to prevent duplication
used_images = set()

# Process each image in the dataframe
for index, row in df.iterrows():
    image_id = row['image_id']
    is_healthy = row['healthy']

    source_path = os.path.join(source_dir, f"{image_id}.jpg")

    # Skip if file doesn't exist
    if not os.path.exists(source_path):
        print(f"Warning: Image {image_id} not found in source directory")
        continue

    # Ensure images are not duplicated
    if image_id in used_images:
        continue

    # Assign images to training set
    if is_healthy == 1 and train_healthy_count < 416:
        dest_path = os.path.join(trainA_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        train_healthy_count += 1
        used_images.add(image_id)

    elif is_healthy == 0 and not train_sick_selected:
        dest_path = os.path.join(trainB_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        train_sick_selected = True
        used_images.add(image_id)

    # Assign images to test set (without duplicating from train set)
    elif is_healthy == 1 and test_healthy_count < 100 and image_id not in used_images:
        dest_path = os.path.join(testA_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        test_healthy_count += 1
        used_images.add(image_id)

    elif is_healthy == 0 and test_sick_count < 81 and image_id not in used_images:
        dest_path = os.path.join(testB_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        test_sick_count += 1
        used_images.add(image_id)

    # Stop early if all constraints are met
    if train_healthy_count == 416 and train_sick_selected and test_healthy_count == 100 and test_sick_count == 81:
        break

print(f"Training: Copied {train_healthy_count} healthy images to {trainA_dir}")
print(f"Training: Copied {'1' if train_sick_selected else '0'} sick image to {trainB_dir}")
print(f"Testing: Copied {test_healthy_count} healthy images to {testA_dir}")
print(f"Testing: Copied {test_sick_count} sick images to {testB_dir}")
print("Dataset preparation completed!")


Training: Copied 416 healthy images to plant_pathology_data/trainA
Training: Copied 1 sick image to plant_pathology_data/trainB
Testing: Copied 100 healthy images to plant_pathology_data/testA
Testing: Copied 81 sick images to plant_pathology_data/testB
Dataset preparation completed!


In [ ]:
# Deleting unnecessary files and folders

dataset_path = "plant_pathology_data/"

keep_dirs = {"trainA", "trainB", "testA", "testB"}

for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)

    if item not in keep_dirs:
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)  # Delete directory
        else:
            os.remove(item_path)  # Delete file

print("Cleanup complete. Only trainA, trainB, testA, and testB remain.")


Cleanup complete. Only trainA, trainB, testA, and testB remain.


In [ ]:
# Resize all images in a given folder to 288x288
def resize_images_in_folder(folder_path, size=(288, 288)):
    if not os.path.exists(folder_path):
        print(f"Skipping {folder_path}, folder does not exist.")
        return

    for filename in os.listdir(folder_path):
        img_path = os.path.join(folder_path, filename)

        # Check if it's an image file
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = cv2.imread(img_path)
            if img is not None:
                resized_img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
                cv2.imwrite(img_path, resized_img)  # Overwrite the original file
                print(f"Resized: {img_path}")

folders = ["trainA", "trainB", "testA", "testB"]

for folder in folders:
    resize_images_in_folder(os.path.join(dataset_path, folder))

print("All images resized to 288x288!")


Resized: plant_pathology_data/trainA/Train_1799.jpg
Resized: plant_pathology_data/trainA/Train_135.jpg
Resized: plant_pathology_data/trainA/Train_1693.jpg
Resized: plant_pathology_data/trainA/Train_1666.jpg
Resized: plant_pathology_data/trainA/Train_551.jpg
Resized: plant_pathology_data/trainA/Train_339.jpg
Resized: plant_pathology_data/trainA/Train_1085.jpg
Resized: plant_pathology_data/trainA/Train_220.jpg
Resized: plant_pathology_data/trainA/Train_307.jpg
Resized: plant_pathology_data/trainA/Train_526.jpg
Resized: plant_pathology_data/trainA/Train_1541.jpg
Resized: plant_pathology_data/trainA/Train_720.jpg
Resized: plant_pathology_data/trainA/Train_1608.jpg
Resized: plant_pathology_data/trainA/Train_1381.jpg
Resized: plant_pathology_data/trainA/Train_111.jpg
Resized: plant_pathology_data/trainA/Train_1683.jpg
Resized: plant_pathology_data/trainA/Train_782.jpg
Resized: plant_pathology_data/trainA/Train_1328.jpg
Resized: plant_pathology_data/trainA/Train_427.jpg
Resized: plant_patholo

# Training the model

In [ ]:
%cd ..

/content/drive/MyDrive/4.3. CycleGAN/pytorch-CycleGAN-and-pix2pix


In [ ]:
torch.backends.cudnn.benchmark = True

In [ ]:
!python train.py --dataroot ./datasets/plant_pathology_data --name sick_leaf_cyclegan --model cycle_gan --gpu_ids 0

----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: ./datasets/plant_pathology_data	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
              display_env: main                          
             display_freq: 400                           
               display_id: 1                             
            display_ncols: 4                             
             display_port: 8097                          
           display_server: http://localhost              
          display_winsize: 256                           
                    epoch: latest                        
             

### Cleaning up the directory

In [ ]:
!mv ./datasets/plant_pathology_data/testA ./datasets/plant_pathology_data
!mv ./datasets/plant_pathology_data/testB ./datasets/plant_pathology_data


In [ ]:
os.makedirs("./datasets/plant_pathology_data/testA", exist_ok=True)
os.makedirs("./datasets/plant_pathology_data/testB", exist_ok=True)

In [ ]:
!cp -r ./datasets/plant_pathology_data/trainA/* ./datasets/plant_pathology_data/testA/
!cp -r ./datasets/plant_pathology_data/trainB/* ./datasets/plant_pathology_data/testB/


# Generating sick leaf images

In [ ]:
!python test.py --dataroot ./datasets/plant_pathology_data \
    --name sick_leaf_cyclegan \
    --model cycle_gan --phase test --no_dropout --num_test 500

----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: ./datasets/plant_pathology_data	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                  gpu_ids: 0                             
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 3                             
                  isTrain: False                         	[default: None]
                load_iter: 0                           

In [ ]:
source_dir = "results/sick_leaf_cyclegan/test_latest/images"
dataset_dir = "datasets/train"
trainA_dir = os.path.join(dataset_dir, "trainA")
trainB_dir = os.path.join(dataset_dir, "trainB")

os.makedirs(trainA_dir, exist_ok=True)
os.makedirs(trainB_dir, exist_ok=True)

count = 0

for filename in os.listdir(source_dir):
    file_path = os.path.join(source_dir, filename)

    if "fake_B" in filename:
        shutil.move(file_path, os.path.join(trainB_dir, filename))
        count +=1

print(f"{count} images moved successfully!")


416 images moved successfully!


In [ ]:
!cp -r ./datasets/plant_pathology_data/trainA/* ./datasets/train/trainA/

# Classifiers

Training and testing ResNet-18 and VGG16 with augmented data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dir = "datasets/train"
test_dir = "datasets/test"

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize the image
])


test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator(device=device))
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, generator=torch.Generator(device=device))

# Print class names
print(f"Classes: {train_dataset.classes}")  # Should be ['trainA', 'trainB']


Classes: ['trainA', 'trainB']


In [ ]:
# Load Pretrained ResNet-18
model = models.resnet18(pretrained=True)

# Modify the last layer for binary classification (Healthy vs. Sick)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # 2 output classes

# Move model to GPU if available
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Print model structure
print(model)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

scheduler = CosineAnnealingLR(optimizer, T_max=30)

def train_model_with_checkpoints(model, optimizer, scheduler, model_name, num_epochs=90, save_interval=20):
    """Train the model and save checkpoints every `save_interval` epochs."""

    start_epoch = 0  # Default to 0 if no checkpoint exists
    checkpoint_path = f"{model_name}_checkpoint.pth"

    # Check if a checkpoint exists and load it
    if os.path.exists(checkpoint_path):
        print(f"Loading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        start_epoch = checkpoint['epoch'] + 1  # Resume from next epoch
        print(f"Resuming training from epoch {start_epoch}")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        print(f"{model_name} - Epoch {epoch+1}/{num_epochs} - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {correct/total:.4f}")

        # **Save checkpoint every `save_interval` epochs**
        if (epoch + 1) % save_interval == 0:
            checkpoint = {
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict()
            }
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved at epoch {epoch+1}.")

    # Save final trained model
    torch.save(model.state_dict(), f"{model_name}_final.pth")
    print(f"Final {model_name} model saved successfully!")



In [ ]:
train_model_with_checkpoints(model, optimizer, scheduler, "ResNet18", num_epochs=30, save_interval=10)

ResNet18 - Epoch 1/30 - Loss: 0.0965 - Accuracy: 0.9651
ResNet18 - Epoch 2/30 - Loss: 0.0525 - Accuracy: 0.9856
ResNet18 - Epoch 3/30 - Loss: 0.0204 - Accuracy: 0.9904
ResNet18 - Epoch 4/30 - Loss: 0.0103 - Accuracy: 0.9976
ResNet18 - Epoch 5/30 - Loss: 0.0132 - Accuracy: 0.9940
ResNet18 - Epoch 6/30 - Loss: 0.0023 - Accuracy: 1.0000
ResNet18 - Epoch 7/30 - Loss: 0.0087 - Accuracy: 0.9976
ResNet18 - Epoch 8/30 - Loss: 0.0012 - Accuracy: 1.0000
ResNet18 - Epoch 9/30 - Loss: 0.0038 - Accuracy: 0.9988
ResNet18 - Epoch 10/30 - Loss: 0.0143 - Accuracy: 0.9976
Checkpoint saved at epoch 10.
ResNet18 - Epoch 11/30 - Loss: 0.0034 - Accuracy: 0.9988
ResNet18 - Epoch 12/30 - Loss: 0.0031 - Accuracy: 0.9988
ResNet18 - Epoch 13/30 - Loss: 0.0014 - Accuracy: 1.0000
ResNet18 - Epoch 14/30 - Loss: 0.0071 - Accuracy: 0.9988
ResNet18 - Epoch 15/30 - Loss: 0.0014 - Accuracy: 1.0000
ResNet18 - Epoch 16/30 - Loss: 0.0019 - Accuracy: 0.9988
ResNet18 - Epoch 17/30 - Loss: 0.0017 - Accuracy: 1.0000
ResNet18 -

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        # Store results
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")

conf_matrix = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", conf_matrix)

print("Classification Report:\n", classification_report(all_labels, all_preds, target_names=train_dataset.classes))


Test Accuracy: 0.5525
Confusion Matrix:
 [[100   0]
 [ 81   0]]
Classification Report:
               precision    recall  f1-score   support

      trainA       0.55      1.00      0.71       100
      trainB       0.00      0.00      0.00        81

    accuracy                           0.55       181
   macro avg       0.28      0.50      0.36       181
weighted avg       0.31      0.55      0.39       181



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
vgg16 = models.vgg16(pretrained=True)
vgg16.classifier[6] = nn.Linear(vgg16.classifier[6].in_features, 2)  # Modify last layer
vgg16 = vgg16.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_vgg = optim.Adam(vgg16.parameters(), lr=0.01)
scheduler_vgg = optim.lr_scheduler.CosineAnnealingLR(optimizer_vgg, T_max=90)
train_model_with_checkpoints(vgg16, optimizer_vgg, scheduler_vgg, "VGG16", num_epochs=90, save_interval=10)


VGG16 - Epoch 1/90 - Loss: 55151894174.2177 - Accuracy: 0.4868
VGG16 - Epoch 2/90 - Loss: 1630648.6271 - Accuracy: 0.4844
VGG16 - Epoch 3/90 - Loss: 80419.1952 - Accuracy: 0.4964
VGG16 - Epoch 4/90 - Loss: 183289.5508 - Accuracy: 0.4832
VGG16 - Epoch 5/90 - Loss: 33132.5499 - Accuracy: 0.4796
VGG16 - Epoch 6/90 - Loss: 4081.6072 - Accuracy: 0.5036
VGG16 - Epoch 7/90 - Loss: 94290.5613 - Accuracy: 0.5048
VGG16 - Epoch 8/90 - Loss: 51040.1091 - Accuracy: 0.5012
VGG16 - Epoch 9/90 - Loss: 758.1086 - Accuracy: 0.5048
VGG16 - Epoch 10/90 - Loss: 995.7650 - Accuracy: 0.4988
Checkpoint saved at epoch 10.
VGG16 - Epoch 11/90 - Loss: 14.6709 - Accuracy: 0.4952
VGG16 - Epoch 12/90 - Loss: 2.0928 - Accuracy: 0.5012
VGG16 - Epoch 13/90 - Loss: 2954.9193 - Accuracy: 0.4784
VGG16 - Epoch 14/90 - Loss: 3.1336 - Accuracy: 0.4796
VGG16 - Epoch 15/90 - Loss: 1.3623 - Accuracy: 0.4796
VGG16 - Epoch 16/90 - Loss: 0.6933 - Accuracy: 0.5000
VGG16 - Epoch 17/90 - Loss: 61.2483 - Accuracy: 0.4880
VGG16 - Epoc

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

vgg16.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = vgg16(images)
        _, preds = torch.max(outputs, 1)

        # Store results
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")

conf_matrix = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", conf_matrix)

print("Classification Report:\n", classification_report(all_labels, all_preds, target_names=train_dataset.classes))


Test Accuracy: 0.5525
Confusion Matrix:
 [[100   0]
 [ 81   0]]
Classification Report:
               precision    recall  f1-score   support

      trainA       0.55      1.00      0.71       100
      trainB       0.00      0.00      0.00        81

    accuracy                           0.55       181
   macro avg       0.28      0.50      0.36       181
weighted avg       0.31      0.55      0.39       181



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
